# Gemma-4 Medical Vision-Language VQA Fine-Tuning with Unsloth

Notebook này hướng dẫn tinh chỉnh (fine-tune) mô hình ngôn ngữ hình ảnh **Gemma-4 E2B/E4B** trên tập dữ liệu **VQA-RAD** để thực hiện tác vụ Medical Visual Question Answering (VQA).
Hệ thống tích hợp giám sát tự động bằng **Comet ML**, thông báo tiến trình huấn luyện thời gian thực thông qua **Discord Webhook** và đăng nhập **Hugging Face** để lưu trữ model online.

*   **Mô hình mặc định:** `unsloth/gemma-4-E2B-it` (có cấu hình sẵn cho `unsloth/gemma-4-E4B-it` ở dạng comment/ẩn đi).
*   **Tập dữ liệu:** `flaviagiammarino/vqa-rad` (2,248 QA pairs, 315 radiology images).
*   **Cơ chế checkpoint:** Lưu vết checkpoint định kỳ, cho phép tiếp tục huấn luyện (resume) từ checkpoint gần nhất nếu bị ngắt quãng.
*   **Công nghệ tối ưu:** Unsloth & SFTTrainer giúp giảm VRAM cực thấp.

In [ ]:
%%capture
import os, re
# Cài đặt thư viện Unsloth cùng các dependency cần thiết
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Chạy local hoặc các dịch vụ đám mây khác
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install --no-deps transformers==5.5.0 "tokenizers>=0.22.0,<=0.23.0"
!pip install torchcodec
!pip install --no-deps --upgrade timm
!pip install comet-ml requests pillow python-dotenv

In [ ]:
import os
import comet_ml
from huggingface_hub import login

# 1. Tải Comet ML API Key, Discord Webhook & HF Token từ Kaggle Secrets hoặc file .env
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    os.environ["COMET_API_KEY"] = user_secrets.get_secret("COMET_API_KEY")
    DISCORD_WEBHOOK_URL = user_secrets.get_secret("DISCORD_WEBHOOK_URL")
    os.environ["HF_TOKEN"] = user_secrets.get_secret("HF_TOKEN")
    print("🔑 Đã nạp thành công API Key, Webhook & HF Token từ Kaggle Secrets!")
except Exception as e:
    # Nếu chạy local hoặc không dùng Kaggle Secrets, thử đọc từ file .env
    try:
        from dotenv import load_dotenv
        # Tự động tìm và nạp file .env từ thư mục hiện hành hoặc thư mục cha
        load_dotenv()
        if "COMET_API_KEY" in os.environ:
            os.environ["COMET_API_KEY"] = os.environ.get("COMET_API_KEY")
        if "HF_TOKEN" in os.environ:
            os.environ["HF_TOKEN"] = os.environ.get("HF_TOKEN")
        DISCORD_WEBHOOK_URL = os.environ.get("DISCORD_WEBHOOK_URL")
        print("🔑 Đã nạp thành công cấu hình giám sát & token từ file .env!")
    except Exception:
        # Fallback điền tay nếu không cấu hình được môi trường
        os.environ["COMET_API_KEY"] = "YOUR_COMET_API_KEY"
        DISCORD_WEBHOOK_URL = "YOUR_DISCORD_WEBHOOK_URL"
        os.environ["HF_TOKEN"] = "YOUR_HF_TOKEN"
        print("⚠️ Không dùng Kaggle Secrets hay .env. Vui lòng kiểm tra điền tay trong code.")

# 2. Đăng nhập Hugging Face Hub
hf_token = os.environ.get("HF_TOKEN")
if hf_token and hf_token != "YOUR_HF_TOKEN" and hf_token.strip() != "":
    login(token=hf_token)
    print("🤗 Đăng nhập Hugging Face Hub thành công!")
else:
    print("⚠️ Thiếu HF_TOKEN. Bạn sẽ không thể push model lên Hugging Face Hub (online).")

# 3. Cấu hình dự án và Khởi tạo Thí nghiệm Comet ML chính thức
os.environ["COMET_PROJECT_NAME"] = "gemma4-medical-vqa"

experiment = comet_ml.Experiment(
    project_name="gemma4-medical-vqa",
    auto_metric_logging=True,
    auto_param_logging=True,
    auto_histogram_logging=True,
)
print("☄️ Comet ML Experiment đã được thiết lập và kích hoạt thành công!")

## 1. Tải và Khám phá Dữ liệu (Data Prep & Processing)

Chúng ta tải dữ liệu `flaviagiammarino/vqa-rad` từ Hugging Face. Tập dữ liệu này chứa sẵn các cặp câu hỏi-trả lời y khoa (`question` và `answer`) kèm hình ảnh chẩn đoán X-ray tương ứng, đã được chia sẵn làm tập train và test.

In [ ]:
from datasets import load_dataset

# Tải dataset train và test
train_dataset = load_dataset("flaviagiammarino/vqa-rad", split="train")
test_dataset = load_dataset("flaviagiammarino/vqa-rad", split="test")

print("📊 Thông tin Train dataset:")
print(train_dataset)
print("\n📊 Thông tin Test dataset:")
print(test_dataset)
print(f"\n👉 Tổng số lượng mẫu: Train ({len(train_dataset)}), Test ({len(test_dataset)})")

In [ ]:
# Trực quan mẫu dữ liệu thứ 3 từ tập train
sample_idx = 2
sample = train_dataset[sample_idx]
print(f"🔍 Mẫu train index {sample_idx}:")
print(f"• Câu hỏi (Question): {sample['question']}")
print(f"• Trả lời (Answer): {sample['answer']}")

# Hiển thị hình ảnh X-ray tương ứng
display(sample["image"].resize((300, 300)))

Mô hình ngôn ngữ hình ảnh (VLM) của Unsloth Gemma-4 yêu cầu dữ liệu chuẩn hóa dạng hội thoại (`messages`). Cấu trúc gồm ảnh và câu hỏi ở role `user`, câu trả lời ở role `assistant`:
```python
[
    {
        "role": "user",
        "content": [
            {"type": "text", "text": sample["question"]},
            {"type": "image", "image": sample["image"]}
        ]
    },
    {
        "role": "assistant",
        "content": [
            {"type": "text", "text": sample["answer"]}
        ]
    }
]
```

In [ ]:
def convert_to_conversation(sample):
    conversation = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": sample["question"]},
                {"type": "image", "image": sample["image"]}
            ]
        },
        {
            "role": "assistant",
            "content": [{"type": "text", "text": sample["answer"]}]
        }
    ]
    return {"messages": conversation}

# Tiến hành ánh xạ toàn bộ tập dữ liệu train và test
converted_train_dataset = [convert_to_conversation(item) for item in train_dataset]
converted_test_dataset = [convert_to_conversation(item) for item in test_dataset]
print(f"✅ Đã xử lý {len(converted_train_dataset)} mẫu train và {len(converted_test_dataset)} mẫu test.")
print("👉 Xem thử mẫu train đầu tiên:")
import pprint
pprint.pprint(converted_train_dataset[0]["messages"])

## 2. Khởi tạo Mô hình Gemma-4 (Model & LoRA Setup)

Chúng ta load model Gemma-4. Theo yêu cầu, mô hình **Gemma-4 E2B** được chạy mặc định, còn mô hình lớn hơn **Gemma-4 E4B** được ẩn đi dưới dạng comment để dễ dàng kích hoạt khi cần.

In [ ]:
from unsloth import FastVisionModel
import torch

# Chọn 1 trong 2 model. Model 4B được ẩn đi (comment out).
model_name = "unsloth/gemma-4-E2B-it"  # Mô hình E2B (2 Tỷ tham số) - Nhẹ, debug nhanh
# model_name = "unsloth/gemma-4-E4B-it"  # Mô hình E4B (4 Tỷ tham số) - Hiệu năng cao hơn

print(f"🚀 Đang tải mô hình từ HuggingFace: {model_name}...")

model, processor = FastVisionModel.from_pretrained(
    model_name,
    load_in_4bit = True,  # Sử dụng định dạng 4bit để siêu tiết kiệm VRAM
    use_gradient_checkpointing = "unsloth",
)

In [ ]:
# Cấu hình LoRA Adapter tiết kiệm tham số (chỉ train ~1.1% tổng trọng số)
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = True,  # Fine-tune module hình ảnh (Vision encoder)
    finetune_language_layers   = True,  # Fine-tune module ngôn ngữ (LLM)
    finetune_attention_modules = True,  # Fine-tune các lớp attention
    finetune_mlp_modules       = True,  # Fine-tune các lớp MLP

    r = 16,                            # Độ rank của LoRA
    lora_alpha = 16,                   # Alpha scale
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_rslora = False,
    target_modules = "all-linear",     # Áp dụng lên toàn bộ linear layers để chất lượng tối ưu
)

In [ ]:
# Áp dụng template chat chuẩn của Gemma-4
from unsloth import get_chat_template

processor = get_chat_template(
    processor,
    chat_template = "gemma-4"
)

## 3. Cấu hình Callback Giám sát gửi thông báo về Discord

Chúng ta định nghĩa lớp callback tùy chỉnh `DiscordWebhookCallback` kế thừa từ `TrainerCallback` của Hugging Face. Khi Trainer chạy qua các mốc bắt đầu, log loss định kỳ và kết thúc, tin nhắn sẽ tự động bắn về kênh Discord thông qua Webhook URL.

In [ ]:
import requests
from transformers import TrainerCallback

class DiscordWebhookCallback(TrainerCallback):
    def __init__(self, webhook_url, model_name):
        self.webhook_url = webhook_url
        self.model_name = model_name

    def send_discord_message(self, content):
        if not self.webhook_url or "discord.com/api/webhooks" not in self.webhook_url:
            print("⚠️ Discord Webhook chưa được thiết lập chính xác hoặc bị bỏ trống. Bỏ qua gửi thông báo!")
            return
        try:
            payload = {"content": content}
            response = requests.post(self.webhook_url, json=payload)
            if response.status_code != 204:
                print(f"⚠️ Gửi thông báo đến Discord thất bại: HTTP {response.status_code}")
        except Exception as e:
            print(f"❌ Lỗi kết nối khi gửi Discord: {e}")

    def on_train_begin(self, args, state, control, **kwargs):
        self.send_discord_message(
            f"🚀 **[BẮT ĐẦU TRAINING]**\n"
            f"• **Mô hình:** `{self.model_name}`\n"
            f"• **Tham số:** `lr={args.learning_rate}`, `steps={args.max_steps}`, `batch_size={args.per_device_train_batch_size}`\n"
            f"• **Platform:** Comet ML Project: `gemma4-medical-vqa`"
        )

    def on_log(self, args, state, control, logs=None, **kwargs):
        # Định kỳ mỗi 10 steps sẽ gửi thông báo cập nhật một lần tránh bị Discord rate-limit
        if logs and "loss" in logs:
            step = state.global_step
            loss = logs["loss"]
            epoch = logs.get("epoch", 0.0)
            lr = logs.get("learning_rate", 0.0)
            if step % 10 == 0 or step == args.max_steps:
                self.send_discord_message(
                    f"📈 **[TIẾN TRÌNH TRAINING]** Step `{step}`/`{args.max_steps}` (Epoch {epoch:.2f})\n"
                    f"• **Loss:** `{loss:.4f}`\n"
                    f"• **Learning Rate:** `{lr:.2e}`"
                )

    def on_train_end(self, args, state, control, **kwargs):
        self.send_discord_message(
            f"🎉 **[TRAINING HOÀN TẤT]**\n• Mô hình `{self.model_name}` đã train thành công với `{state.global_step}` steps!"
        )

## 4. Huấn luyện Mô hình (Fine-Tuning Loop với Checkpoint)

Sử dụng `SFTTrainer` từ TRL cùng collator dữ liệu vision chuyên biệt `UnslothVisionDataCollator` của Unsloth để bắt đầu training. Thiết lập lưu checkpoint định kỳ giúp khôi phục phiên huấn luyện nếu xảy ra sự cố.

In [ ]:
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

# Khởi tạo callback Discord gửi webhook
discord_cb = DiscordWebhookCallback(webhook_url=DISCORD_WEBHOOK_URL, model_name=model_name)

trainer = SFTTrainer(
    model = model,
    train_dataset = converted_train_dataset,
    processing_class = processor.tokenizer,
    data_collator = UnslothVisionDataCollator(model, processor),
    callbacks = [discord_cb],
    args = SFTConfig(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        max_grad_norm = 0.3,
        warmup_ratio = 0.03,
        # max_steps = 60,         # Thiết lập 60 steps để demo nhanh, hãy đặt num_train_epochs=1 nếu train đầy đủ
        num_train_epochs=1,
        learning_rate = 2e-4,
        logging_steps = 1,
        
        # --- CẤU HÌNH CHECKPOINT --- 
        save_strategy = "steps",
        save_steps = 10,        # Lưu checkpoint sau mỗi 10 steps huấn luyện
        save_total_limit = 2,   # Giới hạn giữ tối đa 2 checkpoint mới nhất để tránh đầy bộ nhớ đĩa trên Kaggle
        
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "outputs",
        report_to = "comet",    # Comet ML sẽ tự động bắt sự kiện và hiển thị biểu đồ loss!

        # Các thiết lập bắt buộc phục vụ vision fine-tuning:
        remove_unused_columns = False,
        dataset_text_field = "",
        dataset_kwargs = {"skip_prepare_dataset": True},
        max_length = 2048,
    )
)

In [ ]:
# Kiểm tra bộ nhớ VRAM của GPU hiện hành trước khi chạy
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"🖥️ Thiết bị GPU: {gpu_stats.name} | Dung lượng RAM tối đa: {max_memory} GB.")
print(f"📊 VRAM đã chiếm dụng trước khi train: {start_gpu_memory} GB.")

In [ ]:
# Bắt đầu huấn luyện mô hình
# Để tiếp tục huấn luyện (resume) từ checkpoint gần nhất, uncomment dòng dưới đây:
# trainer_stats = trainer.train(resume_from_checkpoint = True)

# Huấn luyện bình thường từ đầu:
trainer_stats = trainer.train()

## 5. Kiểm thử Mô hình & Lưu trữ LoRA Weights (Inference & Save)

Sau khi train xong, chúng ta chạy thử dự đoán trên một ảnh X-ray y tế mẫu bất kỳ và lưu lại các file adapter LoRA đã được tối ưu lên Hugging Face Hub (online).

In [ ]:
import random

# Chọn 1 mẫu bất kỳ trong dataset test để kiểm tra
test_sample = test_dataset[random.randint(0, len(test_dataset)-1)]
image = test_sample["image"].convert("RGB")
question = test_sample["question"]
gold_answer = test_sample["answer"]

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": question}
        ]
    }
]

input_text = processor.apply_chat_template(messages, add_generation_prompt = True)
inputs = processor(
    image,
    input_text,
    add_special_tokens = False,
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(processor.tokenizer, skip_prompt = True)

print("❓ Câu hỏi (Question):")
print(question)
print("\n📝 Bệnh án gốc (Ground Truth):")
print(gold_answer)
print("\n📝 Mô tả dự đoán từ mô hình sau Fine-tuning:")
_ = model.generate(
    **inputs, 
    streamer = text_streamer, 
    max_new_tokens = 128,
    use_cache = True,
    temperature = 1.0,
    top_p = 0.95,
    top_k = 64
)

In [ ]:
# 1. Lưu adapter LoRA xuống thư mục cục bộ
model.save_pretrained("gemma_4_lora")
processor.save_pretrained("gemma_4_lora")
print("✅ Lưu LoRA adapter thành công vào thư mục 'gemma_4_lora'!")

# 2. Đẩy model lên Hugging Face Hub (online) sử dụng HF Token đã nạp ở đầu notebook
if os.environ.get("HF_TOKEN") and os.environ.get("HF_TOKEN") != "YOUR_HF_TOKEN":
    try:
        # Vui lòng thay thế 'username/model_name' bằng tài khoản HF thực tế của bạn
        # model.push_to_hub("your_name/gemma_4_lora", token = os.environ["HF_TOKEN"])
        # processor.push_to_hub("your_name/gemma_4_lora", token = os.environ["HF_TOKEN"])
        print("🤗 Đã sẵn sàng các lệnh push_to_hub! (Uncomment các lệnh trên để đẩy online)")
    except Exception as e:
        print(f"⚠️ Gặp lỗi khi đẩy model lên HF Hub: {e}")
else:
    print("⚠️ Không tìm thấy HF_TOKEN hợp lệ. Bỏ qua việc upload lên Hugging Face Hub.")